In [ ]:
pip install sentence-transformers pandas


In [ ]:
# 1. 패키지 불러오기 (설치가 안 되었다면 !pip install sentence-transformers 실행)
from sentence_transformers import SentenceTransformer
import pandas as pd

# 2. 모델 로드 (다국어 지원 모델 - 한글 검색을 위해 필수!)
# [왜 이 모델인가요?]: 한국어와 영어를 동시에 이해하는 똑똑한 AI 모델입니다.
print("🧠 AI 모델 로딩 중... 잠시만 기다려주세요.")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("✅ 모델 로드 완료!")

# 3. 테스트 데이터
test_games = [
    "위쳐 3: 괴물 사냥꾼이 되어 모험을 떠나는 다크 판타지 RPG",
    "스타듀 밸리: 시골 마을에서 농사를 짓고 힐링하는 게임",
    "사이버펑크 2077: 미래 도시에서 펼쳐지는 SF 액션"
]

# 4. 임베딩(벡터 변환) 실행
print("🔄 텍스트를 AI 숫자(벡터)로 변환 중...")
embeddings = model.encode(test_games)

# 5. 결과 확인
print(f"▶ 변환된 문장 개수: {len(embeddings)}개")
print(f"▶ 첫 번째 벡터 샘플 (5개 숫자만): {embeddings[0][:5]}")

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 1. 유저의 검색어 임베딩 (영어가 아니어도 됨!)
user_query = "괴물을 사냥하는 어두운 분위기 모험"
query_vector = model.encode([user_query])

# 2. 게임 데이터들과의 유사도 계산
# [왜 이 기술을 쓰나요?]: 유저 검색어(숫자)와 게임 리스트(숫자)가 
# 얼마나 '의미적으로 닮았는지' 0~1 사이의 점수로 뽑아내기 위해서입니다.
distances = cosine_similarity(query_vector, embeddings)

# 3. 결과 보기 좋게 출력
print(f"🔍 유저 검색어: '{user_query}'")
print("-" * 30)

for i, game in enumerate(test_games):
    score = distances[0][i]
    print(f"🎮 게임: {game}")
    print(f"📊 유사도 점수: {score:.4f}")
    print("-" * 30)

🔍 유저 검색어: '괴물을 사냥하는 어두운 분위기 모험'
------------------------------
🎮 게임: 위쳐 3: 괴물 사냥꾼이 되어 모험을 떠나는 다크 판타지 RPG
📊 유사도 점수: 0.6682
------------------------------
🎮 게임: 스타듀 밸리: 시골 마을에서 농사를 짓고 힐링하는 게임
📊 유사도 점수: 0.4276
------------------------------
🎮 게임: 사이버펑크 2077: 미래 도시에서 펼쳐지는 SF 액션
📊 유사도 점수: 0.4113
------------------------------


In [5]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 오늘 작업할 파트: 데이터의 '질' 높이기 (Data Enrichment) 
    openai에 ai 라벨링을 통해 잘 되는지 확인

In [ ]:
import pandas as pd
from openai import OpenAI

# 1. OpenAI 연결 (키를 직접 넣거나 상위 폴더 경로를 활용해)
client = OpenAI(api_key="여기에_너의_키_입력") 

# 2. 기존 데이터 로드
df = pd.read_csv("../data/hidden_gem_bulk_data.csv")

# 3. AI에게 점수 물어보는 함수
def ask_ai_score(game_name):
    # [왜 이 짓을 하나요?]: RAWG에는 없는 '매니아성' 수치를 우리만의 데이터로 만들기 위해서입니다.
    prompt = f"'{game_name}' 게임이 대중적이지 않지만 특정 팬층에게 강력히 어필하는 '숨겨진 명작'인가요? 1점부터 10점 사이의 숫자로만 답하세요."
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}]
    )
    return int(response.choices[0].message.content.strip())

# 4. 딱 5개만 먼저 테스트 (돈 아끼기!)
print("🚀 AI 라벨링 시작...")
df_sample = df.head(5).copy()
df_sample['mania_score'] = df_sample['name'].apply(ask_ai_score)

print("✅ 결과 확인:")
print(df_sample[['name', 'mania_score']])